# 📊 Notebook 2: PySpark DataFrames & The Structured API
### *Mastering Schemas, Ingestion Formats (Parquet vs CSV), Column Operations, and Spark SQL*

> **Companion Video Reference:** [YouTube: PySpark Tutorial | Full Course](https://www.youtube.com/watch?v=94w6hPk7nkM) by Ansh Lamba & [freeCodeCamp PySpark Course](https://www.youtube.com/watch?v=_C8kWBEw6KU)

---

## 🎯 What Will You Learn in This Notebook?

In the early days of Spark (Spark 1.x), developers wrote code using low-level **RDDs (Resilient Distributed Datasets)**. While powerful, RDDs required writing manual Python lambda functions, and Spark had zero visibility into the structure of your data.

Spark 2.0 introduced the **Structured API (DataFrames & Datasets)**:
- Gives Spark a strict understanding of **Columns** and **Data Types**.
- Unlocks the **Catalyst Optimizer** and **Tungsten binary execution**.
- Makes data operations intuitive, SQL-like, and lightning-fast.

In this notebook, we cover:
1. **The DataFrame Abstraction:** Why DataFrames outperform raw RDDs by orders of magnitude.
2. **Strict Schema Definition:** Using `StructType` and `StructField` (and why `inferSchema=True` is dangerous in production).
3. **Storage Formats Compared:** Why **Apache Parquet** is the gold standard for Big Data vs CSV and JSON.
4. **Core DataFrame Transformations:** `select`, `col`, `withColumn`, `withColumnRenamed`, `drop`, `filter`, and conditional `when/otherwise`.
5. **Handling Missing Data (Nulls & NaNs):** `isNull`, `dropna`, `fillna`, and `coalesce`.
6. **Spark SQL Interoperability:** Creating temporary views and executing SQL queries directly on DataFrames.

---


## 🏗️ 1. Why DataFrames? RDDs vs. DataFrames

```
┌──────────────────────────────────────────────┐
│            RDD (Low-Level API)               │
│ • Unstructured JVM / Python objects          │
│ • Opaque to Spark's Catalyst Optimizer       │
│ • Heavy Python serialization overhead (Py4J) │
│ • Slow: Spark doesn't know column types      │
└──────────────────────────────────────────────┘
                       ▲
                       │ (Replaced by)
                       ▼
┌──────────────────────────────────────────────┐
│         DATAFRAME (Structured API)           │
│ • Tabular data with named, typed columns     │
│ • Fully optimized by Catalyst & Tungsten     │
│ • Off-heap binary memory storage (UnsafeRow) │
│ • Identical performance in Python and Scala  │
└──────────────────────────────────────────────┘
```

When you use PySpark DataFrames, your Python code does not process rows one-by-one. Instead, Python communicates your query plan to the JVM via **Py4J**, and the C++/Java Tungsten engine runs it directly on your hardware!


## 📐 2. Defining Explicit Schemas (`StructType` & `StructField`)

### ⚠️ The Production Anti-Pattern: `inferSchema=True`
When reading CSV or JSON files, beginners often write:
```python
spark.read.csv("data.csv", header=True, inferSchema=True) # ❌ DANGEROUS IN PRODUCTION
```
**Why this is dangerous:**
1. **Double Scan Penalty:** To infer types, Spark must read the **entire multi-gigabyte file once** just to guess types, and then read it a second time to load it!
2. **Schema Drift / Silent Bugs:** If an integer column has a single `"N/A"` string on row 5,000,000, Spark silently converts the entire column to `StringType`, breaking downstream jobs.
3. **Best Practice:** Always define an **explicit schema** using `StructType` and `StructField`!


## 🚀 Hands-On Lab: Setting Up Session & Schemas


In [1]:
import os
import sys
from pathlib import Path

# Configure environment
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    BooleanType,
    TimestampType
)
from pyspark.sql.functions import (
    col,
    lit,
    expr,
    when,
    coalesce,
    concat,
    to_date,
    round as spark_round
)

# Initialize local SparkSession
spark = SparkSession.builder \
    .appName("PySpark_DataFrames_and_Structured_API") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ SparkSession initialized successfully!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/22 11:40:40 WARN Utils: Your hostname, Prafull-Mac.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.161 instead (on interface en0)
26/09/22 11:40:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/Users/prafullsaxena/Desktop/Learning/Machine Learning/milvus/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/22 11:40:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession initialized successfully!


### Defining an Explicit Schema
Let's build an explicit schema for an e-commerce customer transactions dataset:


In [2]:
# Define explicit schema
transaction_schema = StructType([
    StructField("order_id", IntegerType(), nullable=False),
    StructField("customer_id", StringType(), nullable=False),
    StructField("customer_name", StringType(), nullable=True),
    StructField("product_category", StringType(), nullable=True),
    StructField("quantity", IntegerType(), nullable=True),
    StructField("unit_price", DoubleType(), nullable=True),
    StructField("country", StringType(), nullable=True),
    StructField("is_loyalty_member", BooleanType(), nullable=True),
])

# Sample raw records (including some realistic null values)
sample_records = [
    (1001, "CUST_01", "Alice Johnson", "Electronics", 2, 499.99, "US", True),
    (1002, "CUST_02", "Bob Smith", "Home & Garden", 5, 24.50, "UK", False),
    (1003, "CUST_03", "Charlie Brown", "Electronics", 1, 1299.00, "US", True),
    (1004, "CUST_04", None, "Books", 3, 15.00, "CA", False),                 # missing name
    (1005, "CUST_05", "Emma Watson", "Home & Garden", None, 45.00, "UK", True), # missing quantity
    (1006, "CUST_06", "Frank Castle", "Electronics", 10, None, "US", False),    # missing price
    (1007, "CUST_07", "Grace Hopper", "Books", 4, 32.50, "US", True),
    (1008, "CUST_08", "Henry Ford", "Automotive", 1, 850.00, None, True),       # missing country
    (1009, "CUST_09", "Ivy Lee", "Books", 2, 19.99, "UK", None),               # missing loyalty
    (1010, "CUST_10", "Jack Ryan", "Electronics", 1, 799.00, "US", False),
]

# Create DataFrame with explicit schema
df = spark.createDataFrame(sample_records, schema=transaction_schema)
print("✅ DataFrame created with explicit schema:")
df.printSchema()
df.show(5, truncate=False)


✅ DataFrame created with explicit schema:
root
 |-- order_id: integer (nullable = false)
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- country: string (nullable = true)
 |-- is_loyalty_member: boolean (nullable = true)



+--------+-----------+-------------+----------------+--------+----------+-------+-----------------+
|order_id|customer_id|customer_name|product_category|quantity|unit_price|country|is_loyalty_member|
+--------+-----------+-------------+----------------+--------+----------+-------+-----------------+
|1001    |CUST_01    |Alice Johnson|Electronics     |2       |499.99    |US     |true             |
|1002    |CUST_02    |Bob Smith    |Home & Garden   |5       |24.5      |UK     |false            |
|1003    |CUST_03    |Charlie Brown|Electronics     |1       |1299.0    |US     |true             |
|1004    |CUST_04    |NULL         |Books           |3       |15.0      |CA     |false            |
|1005    |CUST_05    |Emma Watson  |Home & Garden   |NULL    |45.0      |UK     |true             |
+--------+-----------+-------------+----------------+--------+----------+-------+-----------------+
only showing top 5 rows


## 💾 3. File Formats Compared: Why Parquet Rules Big Data

| Feature | CSV | JSON | Apache Parquet |
| :--- | :--- | :--- | :--- |
| **Storage Layout** | Row-based (text) | Row-based (text) | **Columnar (Binary)** |
| **Schema Included** | ❌ No (all plain text) | ❌ Inferred per line | ✅ **Embedded in metadata footer** |
| **Compression** | Poor (gzip / none) | Poor | ✅ **Snappy / ZSTD / Dictionary (High)** |
| **Column Pruning** | ❌ Must read entire row | ❌ Must parse whole text | ✅ **Reads ONLY selected columns from disk!** |
| **Predicate Pushdown** | ❌ Must scan all bytes | ❌ Must scan all bytes | ✅ **Uses Min/Max statistics in footer to skip blocks!** |

Let's write our DataFrame to Parquet format and inspect how cleanly Spark reads it back:


In [3]:
DATA_DIR = Path("./data/spark_formats")
DATA_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = str(DATA_DIR / "transactions.parquet")

# Save DataFrame as Parquet (overwriting previous run)
df.write.mode("overwrite").parquet(PARQUET_PATH)
print(f"✅ Parquet dataset written to: {PARQUET_PATH}")

# Read Parquet back - notice Spark reads the schema INSTANTLY from the metadata footer!
parquet_df = spark.read.parquet(PARQUET_PATH)
print("✅ Loaded Parquet DataFrame instantly (schema auto-preserved):")
parquet_df.printSchema()


26/09/22 11:40:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✅ Parquet dataset written to: data/spark_formats/transactions.parquet


✅ Loaded Parquet DataFrame instantly (schema auto-preserved):
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- country: string (nullable = true)
 |-- is_loyalty_member: boolean (nullable = true)



## 🛠️ 4. Core DataFrame Transformations

Now let's practice the daily bread-and-butter DataFrame operations:
1. `select` & `col`: Selecting and creating derived projections
2. `withColumn`: Adding new columns or mutating existing ones
3. `withColumnRenamed` & `drop`: Cleaning up column names and removing redundant fields
4. `filter` / `where`: Slicing rows using Boolean predicates
5. `when` / `otherwise`: Conditional SQL-like logic (`CASE WHEN ... THEN ... ELSE ... END`)


In [4]:
# 1. Calculating Total Amount (quantity * unit_price)
# 2. Applying conditional tiering with when/otherwise
# 3. Filtering US and UK transactions
transformed_df = df.select(
    col("order_id"),
    col("customer_id"),
    col("customer_name"),
    col("product_category"),
    col("quantity"),
    col("unit_price"),
    col("country"),
    col("is_loyalty_member")
).withColumn(
    "gross_total",
    spark_round(col("quantity") * col("unit_price"), 2)
).withColumn(
    "order_tier",
    when(col("gross_total") >= 1000.0, lit("High-Value"))
    .when(col("gross_total") >= 200.0, lit("Medium-Value"))
    .otherwise(lit("Standard"))
).filter(
    col("country").isin("US", "UK") & (col("gross_total").isNotNull())
)

print("✅ Transformed and filtered DataFrame:")
transformed_df.show(truncate=False)


✅ Transformed and filtered DataFrame:


+--------+-----------+-------------+----------------+--------+----------+-------+-----------------+-----------+------------+
|order_id|customer_id|customer_name|product_category|quantity|unit_price|country|is_loyalty_member|gross_total|order_tier  |
+--------+-----------+-------------+----------------+--------+----------+-------+-----------------+-----------+------------+
|1001    |CUST_01    |Alice Johnson|Electronics     |2       |499.99    |US     |true             |999.98     |Medium-Value|
|1002    |CUST_02    |Bob Smith    |Home & Garden   |5       |24.5      |UK     |false            |122.5      |Standard    |
|1003    |CUST_03    |Charlie Brown|Electronics     |1       |1299.0    |US     |true             |1299.0     |High-Value  |
|1007    |CUST_07    |Grace Hopper |Books           |4       |32.5      |US     |true             |130.0      |Standard    |
|1009    |CUST_09    |Ivy Lee      |Books           |2       |19.99     |UK     |NULL             |39.98      |Standard    |


## 🧼 5. Handling Missing Data (Nulls & NaNs)

In real-world data pipelines, dirty and incomplete data is inevitable. PySpark provides dedicated functions in `DataFrame.na`:
- `df.na.drop()`: Drops rows containing null values
- `df.na.fill()`: Replaces nulls with default fallback values
- `coalesce(col_a, col_b, default)`: Returns the first non-null value among columns


In [5]:
# Strategy 1: Replace null customer names with 'Guest Customer'
# Strategy 2: Replace null quantities with 1
# Strategy 3: Replace null loyalty flags with False
# Strategy 4: Drop any row that still has a null unit_price
cleaned_df = df.na.fill({
    "customer_name": "Guest Customer",
    "quantity": 1,
    "is_loyalty_member": False,
    "country": "Unknown"
}).na.drop(subset=["unit_price"])

print("✅ Cleaned DataFrame with missing values resolved:")
cleaned_df.show(truncate=False)


✅ Cleaned DataFrame with missing values resolved:


+--------+-----------+--------------+----------------+--------+----------+-------+-----------------+
|order_id|customer_id|customer_name |product_category|quantity|unit_price|country|is_loyalty_member|
+--------+-----------+--------------+----------------+--------+----------+-------+-----------------+
|1001    |CUST_01    |Alice Johnson |Electronics     |2       |499.99    |US     |true             |
|1002    |CUST_02    |Bob Smith     |Home & Garden   |5       |24.5      |UK     |false            |
|1003    |CUST_03    |Charlie Brown |Electronics     |1       |1299.0    |US     |true             |
|1004    |CUST_04    |Guest Customer|Books           |3       |15.0      |CA     |false            |
|1005    |CUST_05    |Emma Watson   |Home & Garden   |1       |45.0      |UK     |true             |
|1007    |CUST_07    |Grace Hopper  |Books           |4       |32.5      |US     |true             |
|1008    |CUST_08    |Henry Ford    |Automotive      |1       |850.0     |Unknown|true     

## 💬 6. Spark SQL: Querying DataFrames with SQL

You don't have to choose between Python syntax and SQL syntax in Spark—**they are 100% interoperable** and compile to the **exact same Catalyst physical plan**!

To query a DataFrame using standard SQL:
1. Register it as a temporary view: `df.createOrReplaceTempView("orders")`
2. Run any standard ANSI SQL query with: `spark.sql("SELECT ... FROM orders")`


In [6]:
# Register cleaned DataFrame as a temporary SQL view
cleaned_df.createOrReplaceTempView("orders")

# Execute an ANSI SQL query with GROUP BY and HAVING
sql_query = '''
    SELECT 
        product_category,
        country,
        COUNT(order_id) AS total_orders,
        SUM(quantity * unit_price) AS total_revenue,
        ROUND(AVG(unit_price), 2) AS avg_unit_price
    FROM orders
    GROUP BY product_category, country
    ORDER BY total_revenue DESC
'''

sql_result_df = spark.sql(sql_query)
print("✅ SQL Query Result on DataFrame:")
sql_result_df.show(truncate=False)


✅ SQL Query Result on DataFrame:


+----------------+-------+------------+-------------+--------------+
|product_category|country|total_orders|total_revenue|avg_unit_price|
+----------------+-------+------------+-------------+--------------+
|Electronics     |US     |3           |3097.98      |866.0         |
|Automotive      |Unknown|1           |850.0        |850.0         |
|Home & Garden   |UK     |2           |167.5        |34.75         |
|Books           |US     |1           |130.0        |32.5          |
|Books           |CA     |1           |45.0         |15.0          |
|Books           |UK     |1           |39.98        |19.99         |
+----------------+-------+------------+-------------+--------------+



### Clean Session Shutdown


In [7]:
# Stop local SparkSession
spark.stop()
print("✅ SparkSession cleanly terminated.")


✅ SparkSession cleanly terminated.


## 📖 Key Takeaways: DataFrames Cheat Sheet

| Operation | PySpark Syntax | SQL Equivalent |
| :--- | :--- | :--- |
| **Select Columns** | `df.select("col1", "col2")` | `SELECT col1, col2` |
| **Filter Rows** | `df.filter(col("age") > 30)` | `WHERE age > 30` |
| **New Column** | `df.withColumn("total", col("q") * col("p"))` | `SELECT (q * p) AS total` |
| **Rename Column** | `df.withColumnRenamed("old", "new")` | `SELECT old AS new` |
| **Conditional Logic**| `when(cond, val).otherwise(default)` | `CASE WHEN cond THEN val ELSE default END` |
| **Drop Nulls** | `df.na.drop(subset=["price"])` | `WHERE price IS NOT NULL` |
| **Fill Nulls** | `df.na.fill({"status": "Pending"})` | `COALESCE(status, 'Pending')` |
| **Create SQL Table** | `df.createOrReplaceTempView("t")` | `CREATE TEMP VIEW t AS ...` |

---
**Next Step:** Move to **`03_advanced_transformations_and_joins.ipynb`** to master aggregations, window functions, and distributed join strategies!
